In [ ]:
!pip install pysam

In [ ]:
import pandas as pd
from tqdm import tqdm
import pysam

In [ ]:
b = 0

In [ ]:
inds_o = {}

with open(f'/mnt/project/DNM/validation/blocks/o_pairs_b{b}.txt') as F:
            
    for i, row in enumerate(F):
                
        row = row.strip()
        row = row.split(' ')
    
        sib = str(row[0])
        o = str(row[1])
        inds_o[sib] = o.split('|')[1:]

In [ ]:
inds_diffs = {}

trios_dnms_df = pd.read_csv('/mnt/project/DNM/trios/out/trios_diffs_info.txt', sep = '\t')

for _, row in trios_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = str(ind_idx.split('_')[0])
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind not in inds_diffs:
        inds_diffs[ind] = set()
    inds_diffs[ind].add(f'{chrom}:{pos}:{ref}:{alt}')

sibs_dnms_df = pd.read_csv('/mnt/project/DNM/sibs/out/sibs_diffs_info_updated.txt', sep = '\t')

for _, row in sibs_dnms_df.iterrows():
    
    ind_idx = row['IID']
    ind = str(ind_idx.split('_')[0])
    
    chrom = int(row['CHROM'])
    pos = int(row['POS'])
    ref = row['ref']
    alt = row['alt']
    
    if ind not in inds_diffs:
        inds_diffs[ind] = set()
    inds_diffs[ind].add(f'{chrom}:{pos}:{ref}:{alt}')

In [ ]:
inds_path = {}

# paths = '/mnt/project/Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]/'

for ind in inds_o:
    
    if ind not in inds_diffs:
        continue
    
    # inds_path[ind] = f'{paths}{ind[:2]}/{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[ind] = f'{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    
    for o in inds_o[ind]:
        # inds_path[o] = f'{paths}{o[:2]}/{o}_24051_0_0.dragen.hard-filtered.gvcf.gz'
        inds_path[o] = f'{o}_24051_0_0.dragen.hard-filtered.gvcf.gz'

In [ ]:
inds_diffs_missing = {}
inds_diffs_transmitted = {}

for _, ind in tqdm(enumerate(inds_o)):
    
    if ind not in inds_diffs:
        continue
    
    for o in inds_o[ind]:
    
        # Offspring --------------------------------

        try:
            vcf = pysam.VariantFile(inds_path[o])
        except:
            continue # failed to read gVCF

        o_info = {}

        for diff_id in inds_diffs[ind]:

            chrom = int(diff_id.split(':')[0])
            pos = int(diff_id.split(':')[1])
            ref = diff_id.split(':')[2]
            alt = diff_id.split(':')[3]

            for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):

                for f in list(F.samples):

                    dp = F.samples[f]['DP']
                    gq = F.samples[f]['GQ']
                    gt = F.samples[f]['GT']

                    if None in gt:
                        continue

                    if F.pos == pos:

                        ref_ad = 0
                        alt_ad = 0

                        if ref in F.alleles:
                            i = F.alleles.index(ref)
                            ref_ad = F.samples[f]['AD'][i]

                        if alt in F.alleles:
                            i = F.alleles.index(alt)
                            alt_ad = F.samples[f]['AD'][i]

                        alleles = (F.alleles[gt[0]], F.alleles[gt[1]])

                    else:
                        ref_ad = dp
                        alt_ad = 0
                        alleles = (ref, ref)

                    o_info[diff_id] = (ref_ad, alt_ad, dp, gq, alleles)

        # Individual --------------------------------

        try:
            vcf = pysam.VariantFile(inds_path[ind])
        except:
            continue # failed to read gVCF

        if ind not in inds_diffs_missing:
            inds_diffs_missing[ind] = []
            inds_diffs_transmitted[ind] = []

        for diff_id in inds_diffs[ind]:

            if diff_id not in o_info:
                continue

            chrom = int(diff_id.split(':')[0])
            pos = int(diff_id.split(':')[1])
            ref = diff_id.split(':')[2]
            alt = diff_id.split(':')[3]

            for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):

                for f in list(F.samples):

                    dp = F.samples[f]['DP']
                    gq = F.samples[f]['GQ']
                    gt = F.samples[f]['GT']

                    if None in gt:
                        continue

                    if F.pos == pos:

                        ref_ad = 0
                        alt_ad = 0

                        if ref in F.alleles:
                            i = F.alleles.index(ref)
                            ref_ad = F.samples[f]['AD'][i]

                        if alt in F.alleles:
                            i = F.alleles.index(alt)
                            alt_ad = F.samples[f]['AD'][i]

                        alleles = (F.alleles[gt[0]], F.alleles[gt[1]])

                    else:
                        ref_ad = dp
                        alt_ad = 0
                        alleles = (ref, ref)

                    info = (diff_id, ref_ad, alt_ad, dp, gq, alleles,
                            o_info[diff_id][0], o_info[diff_id][1], o_info[diff_id][2], o_info[diff_id][3], o_info[diff_id][4])

                    if ((ref in alleles and alt in alleles) and
                        (o_info[diff_id][4][0] == o_info[diff_id][4][1] and ref in o_info[diff_id][4])):
                        inds_diffs_missing[ind].append(info)
                    else:
                        inds_diffs_transmitted[ind].append(info)

In [ ]:
with open(f'o_missing_b{b}.txt', 'w') as f:
    for ind in inds_diffs_missing:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, o_ref_ad, o_alt_ad, o_dp, o_gq, o_alleles) in inds_diffs_missing[ind]:
            f.write(f'{ind} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {o_ref_ad} {o_alt_ad} {o_dp} {o_gq} {o_alleles}\n')

In [ ]:
with open(f'o_transmitted_b{b}.txt', 'w') as f:
    for ind in inds_diffs_transmitted:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, o_ref_ad, o_alt_ad, o_dp, o_gq, o_alleles) in inds_diffs_transmitted[ind]:
            f.write(f'{ind} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {o_ref_ad} {o_alt_ad} {o_dp} {o_gq} {o_alleles}\n')